In [19]:
# Task 1.1

import csv
import json

def parse_row(row):
    return {
        **row,
        "age": int(row["age"]),
        "fee_inr": float(row["fee_inr"]),
        "prior_no_shows": int(row["prior_no_shows"]),
        "distance_km": float(row["distance_km"]),
    }

accepted = []
rejected = []


with open("appointments.csv", newline="", encoding="utf-8") as f:
  reader = csv.DictReader(f)

  headers = reader.fieldnames
  print("Headers:", headers)

  row_count = 0

  for row in reader:
    row_count += 1

    try:
      parsed = parse_row(row)
      accepted.append(parsed)
    except ValueError as e:
      row["_error"] = str(e)
      rejected.append(row)

print("Row count:", row_count)
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted: 988
Rejected: 32


In [20]:
# Task 1.2

import pandas as pd

# Load data
df = pd.read_csv("appointments.csv")

# Initial inspection
print("Shape:")
print(df.shape)

print("\nInfo:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())



Shape:
(1020, 14)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
None

Missing values:
appointment_id      0
patient_id          0
c

In [21]:

# ---------------------------
# Step (a): Drop exact duplicates
# ---------------------------
df = df.drop_duplicates()
print("\nAfter dropping duplicate rows:", df.shape)


After dropping duplicate rows: (1000, 14)


In [22]:
# ---------------------------
# Step (b): Clean booking_channel
# ---------------------------
df["booking_channel"] = (
    df["booking_channel"]
      .str.strip()
      .str.lower()
      .replace({
          "app": "App",
          "phone": "Phone",
          "walk-in": "Walk-In",
          "walk in": "Walk-In",
          "walkin": "Walk-In"
      })
)

print("\nAfter cleaning booking_channel:", df.shape)


After cleaning booking_channel: (1000, 14)


In [23]:
# ---------------------------
# Step (c): Convert appt_date
# ---------------------------
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

print("\nAfter converting appt_date:", df.shape)


After converting appt_date: (1000, 14)


In [24]:
# ---------------------------
# Step (d): Convert booked_on
# ---------------------------
df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce"
)

print("\nAfter converting booked_on:", df.shape)


After converting booked_on: (1000, 14)


In [25]:
# ---------------------------
# Step (e): Drop rows where
# appt_date is missing after conversion
# OR status is missing
# ---------------------------
df = df.dropna(
    subset=["appt_date", "status"]
)

print("\nAfter dropping missing appt_date/status:", df.shape)

print("\nFinal shape:", df.shape)


After dropping missing appt_date/status: (990, 14)

Final shape: (990, 14)


In [26]:
# Task 1.3

import numpy as np
import pandas as pd

# ---------------------------
# (a) Invalid age -> missing
# ---------------------------
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

In [27]:
# ---------------------------
# (b) Fill gaps
# ---------------------------

# age -> median, then convert to int
df["age"] = (
    df["age"]
    .fillna(df["age"].median())
    .round()
    .astype(int)
)

In [28]:
# distance_km -> median
df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

In [29]:
# ---------------------------
# (c) Outlier capping (IQR)
# ---------------------------

def cap_outliers(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    before = df[col].copy()

    df[col] = np.clip(df[col], lower, upper)

    capped_count = (before != df[col]).sum()

    return capped_count

distance_capped = cap_outliers(df, "distance_km")
fee_capped = cap_outliers(df, "fee_inr")

In [30]:
# ---------------------------
# (d) Rounding
# ---------------------------

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"]
    .round()
    .astype(int)
)

In [31]:
# ---------------------------
# Required output
# ---------------------------

print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 26
Fee values capped: 3


In [32]:
# Task 1.4

# ---------------------------
# Feature engineering
# ---------------------------

df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

df["no_show"] = (df["status"] == "No-Show").astype(int)

# ---------------------------
# NumPy matrix
# ---------------------------

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# ---------------------------
# (a) Mean of each column
# ---------------------------

column_means = X.mean(axis=0)

print("Age mean:", column_means[0])
print("Distance mean:", column_means[1])
print("Fee mean:", column_means[2])
print("Lead-days mean:", column_means[3])

# ---------------------------
# (b) Rows with lead_days > 14
# ---------------------------

mask = X[:, 3] > 14

count_gt14 = mask.sum()
share_gt14 = mask.mean()

print("Rows with lead_days > 14:", count_gt14)
print("Share with lead_days > 14:", share_gt14)

# ---------------------------
# (c) Z-scores for distance_km
# ---------------------------

distance = X[:, 1]

z_scores = (distance - distance.mean()) / distance.std()

outlier_count = (np.abs(z_scores) > 2).sum()

print("Rows with |z| > 2:", outlier_count)

Age mean: 48.173737373737374
Distance mean: 9.50090909090909
Fee mean: 635.7141414141414
Lead-days mean: 11.819191919191919
Rows with lead_days > 14: 296
Share with lead_days > 14: 0.298989898989899
Rows with |z| > 2: 53


In [33]:
# Task 1.5

import pandas as pd
import requests

clinics = pd.read_csv("clinics.csv")

temps = []

for _, row in clinics.iterrows():

    temp = None

    for attempt in range(2):
        try:
            r = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            if r.status_code == 200:

                data = r.json()

                temp = data["hourly"]["temperature_2m"][0]
                break

            else:
                raise Exception(f"status {r.status_code}")

        except requests.exceptions.RequestException as e:

            if attempt == 1:
                print(f"{row['clinic_name']}: {e}")

    temps.append(temp)

clinics["temp_c"] = temps

clinics.to_csv(
    "clinics_enriched.csv",
    index=False
)

In [34]:
# Task 1.6

import pandas as pd

# -------------------------
# Check available columns
# -------------------------
print("Columns:")
print(df.columns.tolist())


Columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [35]:
# -------------------------
# Convert dates if present
# -------------------------
for col in ["booked_on", "appt_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")

In [36]:
# -------------------------
# Required output columns
# -------------------------
required_cols = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    print("Missing columns:", missing_cols)
else:

    df_export = df[required_cols].copy()

    print("Rows:", len(df_export))
    print("Columns:", len(df_export.columns))

    print(
        "Distinct appointment_id:",
        df_export["appointment_id"].nunique()
    )

    print(
        "Total missing:",
        df_export.isnull().sum().sum()
    )

    df_export.to_csv(
        "appointments_clean.csv",
        index=False
    )

    print("appointments_clean.csv created")


Rows: 990
Columns: 16
Distinct appointment_id: 990
Total missing: 0
appointments_clean.csv created


In [37]:
# -------------------------
# SQL generation
# -------------------------

try:
    clinics = pd.read_csv("clinics_enriched.csv")

    insert_count = 0

    with open("load_data.sql", "w", encoding="utf-8") as f:

        # Clinics first
        for _, row in clinics.iterrows():

            temp_c = (
                "NULL"
                if pd.isna(row["temp_c"])
                else row["temp_c"]
            )

            f.write(
                f"INSERT INTO clinics VALUES "
                f"('{row['clinic_id']}',"
                f"'{row['clinic_name']}',"
                f"{temp_c});\n"
            )

            insert_count += 1

        # Appointments second
        if not missing_cols:

            for _, row in df_export.iterrows():

                f.write(
                    f"INSERT INTO appointments VALUES ("
                    f"'{row['appointment_id']}',"
                    f"'{row['patient_id']}',"
                    f"'{row['clinic_id']}',"
                    f"'{row['booked_on']}',"
                    f"'{row['appt_date']}',"
                    f"{row['age']},"
                    f"'{row['gender']}',"
                    f"{row['distance_km']},"
                    f"{row['fee_inr']},"
                    f"'{row['booking_channel']}',"
                    f"'{row['visit_type']}',"
                    f"{row['reminder_sent']},"
                    f"{row['prior_no_shows']},"
                    f"'{row['status']}',"
                    f"{row['lead_days']},"
                    f"{row['no_show']}"
                    f");\n"
                )

                insert_count += 1

    print("INSERT statements:", insert_count)

except FileNotFoundError:
    print("clinics_enriched.csv not found")

INSERT statements: 998


In [38]:
# Task3.1

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load cleaned file from Part 1
df = pd.read_csv("appointments_clean.csv")

In [39]:
# -----------------------------------
# Convert reminder_sent Yes/No -> 1/0
# -----------------------------------

df["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

In [40]:
# -----------------------------------
# Features
# -----------------------------------

X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
]

# One-hot encoding
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

In [41]:
# Target
y = df["no_show"]

# -----------------------------------
# Train/Test Split
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [42]:
# -----------------------------------
# Required checks
# -----------------------------------

print("X shape:", X.shape)
print("Number of features:", X.shape[1])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("No-shows in test set:", y_test.sum())

# -----------------------------------
# Baseline model
# Always predict attended (0)
# -----------------------------------

baseline_pred = [0] * len(y_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

print("Baseline accuracy:", baseline_accuracy)

X shape: (990, 11)
Number of features: 11
Train rows: 792
Test rows: 198
No-shows in test set: 49
Baseline accuracy: 0.7525252525252525
